In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../../../data/processed/review_histogram.csv')
df['date_parsed'] = pd.to_datetime(df['date'], unit='s')

games = pd.read_csv('../../../data/processed/steam_stratified_sample.csv')[['appid', 'name_store']]
df = df.merge(games, on='appid', how='left')
df['name_store'] = df['name_store'].fillna(df['appid'].astype(str))

print(df.shape)
print(df.dtypes)

(4997, 8)
appid                           int64
date                            int64
recommendations_up              int64
recommendations_down            int64
rollup_type                       str
data_type                         str
date_parsed             datetime64[s]
name_store                        str
dtype: object


## 데이터 개요

`review_histogram.csv`는 Steam appreviewhistogram API에서 수집한 게임별 리뷰 시계열 데이터다.

| 컬럼 | 설명 |
|------|------|
| `data_type=rollups` | 출시 이후 전체 기간의 월별·주별 집계 버킷 |
| `data_type=recent` | 최근 30일 이내 일별 집계 버킷 |
| `rollup_type` | 버킷 단위 — `day` / `week` / `month` |

분석 구성:
1. 게임별 전체 기간 리뷰 유입 추이 (라인 차트)
2. 게임별 긍정/부정 리뷰 시계열 (개별 스택 바)
3. 게임별 누적 긍정률 비교 (파이 차트)
4. 최근 30일 일별 리뷰 유입 (라인 차트)

In [3]:
rollups = df[df['data_type'] == 'rollups'].copy()
rollups['total'] = rollups['recommendations_up'] + rollups['recommendations_down']

fig = px.line(
    rollups,
    x='date_parsed',
    y='total',
    color='name_store',
    title='게임별 월별/주별 리뷰 유입 추이',
    labels={'date_parsed': '날짜', 'total': '총 리뷰 수', 'name_store': '게임'},
    markers=True
)
fig.update_layout(hovermode='x unified', legend_title_text='게임')
fig.show()

### 해석: 게임별 리뷰 유입 추이

라인 차트는 게임별 월별·주별 총 리뷰 수(긍정+부정)의 시계열을 보여준다.

- **초기 집중형** (출시 직후 피크 후 급감): 마케팅·큐레이션 효과가 단기에 소진된 패턴이다. 이후 유입을 유지하려면 할인 이벤트나 업데이트 공지가 필요하다.
- **장기 분산형** (수개월에 걸친 완만한 유입): 입소문 또는 지속적인 콘텐츠 업데이트가 효과를 내고 있는 패턴이다. Steam 알고리즘 추천에 안정적으로 노출되고 있을 가능성이 높다.
- **복수의 피크**: 특정 시점에 스파이크가 재발한다면 세일·업데이트·스트리머 노출 등 외부 이벤트와 시점을 교차 확인할 필요가 있다.

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math

games_list = rollups['name_store'].unique()
n_cols = 5
n_rows = math.ceil(len(games_list) / n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=games_list,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.08,
    horizontal_spacing=0.05,
)

for i, game in enumerate(games_list):
    row = i // n_cols + 1
    col = i % n_cols + 1
    g = rollups[rollups['name_store'] == game].sort_values('date_parsed')

    fig.add_trace(go.Bar(
        x=g['date_parsed'], y=g['recommendations_up'],
        name='긍정', marker_color='#4C9BE8',
        showlegend=(i == 0),
    ), row=row, col=col)
    fig.add_trace(go.Bar(
        x=g['date_parsed'], y=g['recommendations_down'],
        name='부정', marker_color='#E8604C',
        showlegend=(i == 0),
    ), row=row, col=col)

fig.update_layout(
    title='게임별 긍정/부정 리뷰 시계열 (개별)',
    barmode='stack',
    height=300 * n_rows,
    legend_title_text='리뷰 유형',
)
fig.show()

In [5]:
pie_data = (
    rollups.groupby('name_store')[['recommendations_up', 'recommendations_down']]
    .sum()
    .reset_index()
)
pie_data['positive_rate'] = pie_data['recommendations_up'] / (
    pie_data['recommendations_up'] + pie_data['recommendations_down']
)
pie_data = pie_data.sort_values('positive_rate', ascending=False)

n_cols = 5
n_rows = math.ceil(len(pie_data) / n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    specs=[[{'type': 'pie'}] * n_cols for _ in range(n_rows)],
    subplot_titles=pie_data['name_store'].tolist(),
)

for i, row_data in enumerate(pie_data.itertuples()):
    r = i // n_cols + 1
    c = i % n_cols + 1
    fig.add_trace(go.Pie(
        values=[row_data.recommendations_up, row_data.recommendations_down],
        labels=['긍정', '부정'],
        marker_colors=['#4C9BE8', '#E8604C'],
        textinfo='percent',
        showlegend=(i == 0),
    ), row=r, col=c)

fig.update_layout(
    title='게임별 전체 기간 긍정/부정 리뷰 비율 (긍정률 높은 순)',
    height=280 * n_rows,
    legend_title_text='리뷰 유형',
)
fig.show()